# 🔧 HELIOS — Fine-tune PRO (SFT propre → DPO)

**Objectif :** UN modèle spécialiste fiable des 120 types (bat qwen+RAG), sans la régression du 1er essai.

### Avant de lancer (2 minutes) :
1. Menu **Add Input → Datasets → New Dataset** : uploade `sft_120.jsonl` **et** `dpo_120.jsonl`.
2. **Settings → Accelerator → GPU T4 x1** (gratuit).
3. **Run All**. (~1-2 h). Le `.gguf` sera téléchargeable à la fin.

Config anti-régression prouvée : Qwen2.5-3B, QLoRA r64 rslora, LR SFT 5e-5 / DPO 5e-6, grad clip 1.0.


In [ ]:
# 1) Dépendances
!pip install -q unsloth "trl>=0.11" datasets


In [ ]:
# 2) Chemins des datasets (adapte si besoin au nom de ton dataset Kaggle)
import glob, os
def find(name):
    hits=glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} introuvable — uploade-le dans Add Input'
    return hits[0]
SFT_FILE=find('sft_120.jsonl'); DPO_FILE=find('dpo_120.jsonl')
print('SFT:',SFT_FILE); print('DPO:',DPO_FILE)


In [ ]:
# 3) Modèle de base + LoRA
from unsloth import FastLanguageModel
import torch, re
MAXLEN=1024
model, tok = FastLanguageModel.from_pretrained('unsloth/Qwen2.5-3B-Instruct', max_seq_length=MAXLEN, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=64, lora_alpha=128, lora_dropout=0.0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=42, use_rslora=True)


## Étape 1/2 — SFT sur données PROPRES (format net + 120 actions)


In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
sft = load_dataset('json', data_files=SFT_FILE, split='train').train_test_split(test_size=0.05, seed=42)
def to_text(ex): return {'text': tok.apply_chat_template(ex['messages'], tokenize=False)}
sft_train = sft['train'].map(to_text, remove_columns=sft['train'].column_names)
SFTTrainer(model=model, tokenizer=tok, train_dataset=sft_train, args=SFTConfig(
    per_device_train_batch_size=2, gradient_accumulation_steps=4, num_train_epochs=2,
    learning_rate=5e-5, max_grad_norm=1.0, lr_scheduler_type='cosine', warmup_ratio=0.05,
    optim='adamw_8bit', weight_decay=0.01, max_seq_length=MAXLEN, logging_steps=20, seed=42,
    dataset_text_field='text', output_dir='sft_out', report_to='none')).train()


## Étape 2/2 — DPO (préférer l'action nette à la prose vague)


In [ ]:
from trl import DPOConfig, DPOTrainer
def dpo_fmt(ex):
    p = tok.apply_chat_template(ex['prompt'], tokenize=False, add_generation_prompt=True)
    return {'prompt': p, 'chosen': ex['chosen']+tok.eos_token, 'rejected': ex['rejected']+tok.eos_token}
dpo = load_dataset('json', data_files=DPO_FILE, split='train')
dpo = dpo.map(dpo_fmt, remove_columns=dpo.column_names)
DPOTrainer(model=model, ref_model=None, args=DPOConfig(
    per_device_train_batch_size=2, gradient_accumulation_steps=4, num_train_epochs=1,
    learning_rate=5e-6, beta=0.1, max_length=MAXLEN, max_prompt_length=640, max_grad_norm=1.0,
    lr_scheduler_type='cosine', warmup_ratio=0.05, optim='adamw_8bit', seed=42, logging_steps=10,
    output_dir='dpo_out', report_to='none'), train_dataset=dpo, processing_class=tok).train()


## Éval rapide — action nommée sur held-out (objectif : battre 0.93)


In [ ]:
FastLanguageModel.for_inference(model)
ok=tot=0
for ex in sft['test'].select(range(min(200,len(sft['test'])))):
    g=re.search(r'ACTION=([A-Z_]+)', ex['messages'][-1]['content']); g=g.group(1) if g else None
    p=tok.apply_chat_template(ex['messages'][:-1], tokenize=False, add_generation_prompt=True)
    out=tok.batch_decode(model.generate(**tok(p,return_tensors='pt').to(model.device), max_new_tokens=64, do_sample=False))[0]
    pr=re.search(r'ACTION=([A-Z_]+)', out.split(p)[-1] if p in out else out)
    tot+=1; ok+= 1 if (g and pr and pr.group(1)==g) else 0
print(f'ACTION NOMMÉE held-out = {ok/max(1,tot):.3f}  (>0.93 = déployable)')


## Export GGUF Q4 → à télécharger + déployer avec `deploy_helios_pro.py`


In [ ]:
model.save_pretrained_gguf('helios-120-pro', tok, quantization_method='q4_k_m')
import glob; print('GGUF:', glob.glob('helios-120-pro*/*.gguf') or glob.glob('*.gguf'))
# Le fichier .gguf est dans /kaggle/working -> onglet Output -> Download.
